## Thư viện và hằng số chung

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

Path("../results").mkdir(exist_ok=True)
Path("../figures").mkdir(exist_ok=True)

SEED = 42
R_SERVICE_KM = 1.0      # Bán kính phục vụ của một trạm
D_MIN_KM = 1.0          # Khoảng cách tối thiểu giữa 2 trạm
MAX_NEW_STATIONS = 10   # Số trạm mới tối đa
GRID_STEP_KM = 1.0      # Bước lưới khi sinh ứng viên
COST_PER_STATION = 1.0  # Đơn vị chi phí cho một trạm
COST_PER_KM = 2.0       # Đơn vị chi phí cho một km tuyến
BUDGET = 50.0           # Ngân sách quy ước
DEFAULT_WEIGHTS = {"pop": 1.0, "cost": 0.1, "penalty": 5.0}

## Khối 1: Dữ liệu

1.1 Đọc bảng phường

In [ ]:
wards_raw = pd.read_csv("../data/wards.csv", encoding="utf-8")
display(wards_raw.head())

1.2 Đọc danh sách trạm metro hiện có

In [ ]:
stations_raw = pd.read_csv("../data/stations.csv", encoding="utf-8")
display(stations_raw.head())

1.3 Chuẩn hoá và kiểm tra data

In [ ]:
import unicodedata

def normalize_text(s):
    if pd.isna(s):
        return ""
    return unicodedata.normalize("NFC", str(s)).strip()

def clean_wards(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    df["ten"] = df["ten"].astype(str).apply(normalize_text).str.replace(r"\s+", " ", regex=True)
    df["quan"] = df["quan"].astype("object").fillna("").astype(str).apply(normalize_text)
    df["dan_so"] = pd.to_numeric(df["dan_so"].astype(str).str.replace(r"[.,\s]", "", regex=True), errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    return df

def report_quality(df, name, subset_dup):
    print(f"--- Kiểm tra {name}: {len(df)} dòng ---")
    print("Giá trị thiếu theo cột:\n", df.isna().sum())
    print(f"Dòng trùng {subset_dup}:", df.duplicated(subset=subset_dup).sum())
    # Khung toạ độ mở rộng bao gồm Bình Dương và Vũng Tàu sau sát nhập
    bad = ~df["lat"].between(10.3, 11.5) | ~df["lon"].between(106.3, 107.6)
    print("Toạ độ ngoài khung TP.HCM:", bad.sum())
    if bad.sum() > 0:
        print(df[bad][["ten", "lat", "lon"]])

wards = clean_wards(wards_raw)
report_quality(wards, "wards", ["ten", "quan"])

# Xử lý data trùng / thiếu / lỗi
n_before_w = len(wards)
mask_w_geo = wards["lat"].between(10.3, 11.5) & wards["lon"].between(106.3, 107.6)
wards = wards[mask_w_geo].drop_duplicates(subset=["ten", "quan"])
wards = wards.dropna(subset=["dan_so", "lat", "lon"])
wards = wards[wards["dan_so"] > 0].reset_index(drop=True)
print(f"Wards sau khi làm sạch: {len(wards)}/{n_before_w} dòng (tổng dân số: {wards['dan_so'].sum():,.0f})\n")

# Làm sạch bảng trạm và tách nhóm đã có vs quy hoạch
stations = stations_raw.copy()
stations.columns = stations.columns.str.strip().str.lower()
stations["ten"] = stations["ten"].astype(str).apply(normalize_text)
stations["tuyen"] = stations["tuyen"].astype(str).apply(normalize_text)
stations["trang_thai"] = stations["trang_thai"].astype(str).apply(normalize_text)
stations["lat"] = pd.to_numeric(stations["lat"], errors="coerce")
stations["lon"] = pd.to_numeric(stations["lon"], errors="coerce")

report_quality(stations, "stations", ["ten", "tuyen"])
n_before_s = len(stations)
mask_s_geo = stations["lat"].between(10.3, 11.5) & stations["lon"].between(106.3, 107.6)
stations = stations[mask_s_geo].drop_duplicates(subset=["ten", "tuyen"]).dropna(subset=["lat", "lon"]).reset_index(drop=True)

# Tách trạm đã có và trạm quy hoạch
existing = stations[stations["trang_thai"].isin(["dang_hoat_dong", "dang_xay"])].reset_index(drop=True)
planned = stations[stations["trang_thai"] == "quy_hoach"].reset_index(drop=True)

print(f"=> Station sau làm sach: {len(stations)}/{n_before_s} dòng")
print(f"Số trạm đã có (Hoạt động + đang xây): {len(existing)}")
print(f"Số trạm quy hoạch: {len(planned)}")

1.4 Hàm tính khoảng cách Haversine

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def distance_matrix(lat_a, lon_a, lat_b, lon_b):
    return haversine(np.asarray(lat_a)[:, None], np.asarray(lon_a)[:, None],
                     np.asarray(lat_b)[None, :], np.asarray(lon_b)[None, :])

# Bến Thành -> Suối Tiên
dist_test = haversine(10.77075, 106.69762, 10.87957,106.81417)
print(f"Khoảng cách chim bay Bến Thành - Suối Tiên: {dist_test:.2f} km")
assert abs(haversine(10.0, 106.0, 10.0, 106.0)) < 1e-9

1.5 Vẽ bản đồ trạng thái
- vẽ phân bố dân số các phường/xã làm nền và nối riêng từng tuyến trạm hiện có (existing)

In [ ]:
plt.rcParams.update({
    "figure.figsize": (9, 9), 
    "font.size": 11, 
    "axes.grid": True, 
    "grid.alpha": 0.3
})

def plot_current_map(wards, existing_stations, ax=None, save_path=None):
    if ax is None:
        _, ax = plt.subplots()
    sizes = 300 * wards["dan_so"] / wards["dan_so"].max()
    sc = ax.scatter(
        wards["lon"], wards["lat"], s=sizes, c=wards["dan_so"], 
        cmap="YlOrRd", alpha=0.7, edgecolors="k", linewidth=0.3, 
        label="Phường/Xã (Cỡ/Màu = dân số)"
    )
    plt.colorbar(sc, ax=ax, label="Dân số (người)", shrink=0.8)

    # Vẽ điểm trạm và nối đường theo từng tuyến
    ax.scatter(
        existing_stations["lon"], existing_stations["lat"], 
        marker="s", s=45, c="tab:blue", edgecolor="k",zorder=4, 
        label="Trạm Metro hiện có / Đang xây"
    )
    for _, group in existing_stations.groupby("tuyen"):
        ax.plot(group["lon"], group["lat"], c="tab:blue", lw=1.5, zorder=3)

    ax.set_xlabel("Kinh độ")
    ax.set_ylabel("Vĩ độ")
    ax.set_title("Hiện trạng phân bố dân số và mạng lưới trạm Metro TP.HCM")
    ax.set_aspect(1 / np.cos(np.radians(wards["lat"].mean())))
    ax.legend(loc="upper left")

    if save_path: 
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
    return ax

plot_current_map(wards, existing, save_path="../figures/hien_trang.png")
plt.show()

## Khối 2: Tập ứng viên

2.1 Sinh tập vị trí ứng viên (tâm phường + lưới điểm)
- Kết hợp 2 nguồn ứng viên: tâm các phường/xã (tam_phuong) và lưới điểm đều (luoi) phủ khung tọa độ vùng nghiên cứu với bước lưới GRID_STEP_KM

In [ ]:
def generate_candidates_from_grid(wards, step_km):
    lat_min, lat_max = wards["lat"].min(), wards["lat"].max()
    lon_min, lon_max = wards["lon"].min(), wards["lon"].max()
    dlat = step_km / 111.0
    dlon = step_km / (111.0 * np.cos(np.radians((lat_min + lat_max) / 2)))
    lats = np.arange(lat_min, lat_max + 1e-9, dlat)
    lons = np.arange(lon_min, lon_max + 1e-9, dlon)
    g = np.array([(a, b) for a in lats for b in lons])
    return pd.DataFrame({
        "ten": [f"G{i}" for i in range(len(g))],
        "lat": g[:, 0],
        "lon": g[:, 1],
        "nguon": "luoi"
    })

cand_ward = wards[["ten", "lat", "lon"]].assign(nguon="tam_phuong")
cand_grid = generate_candidates_from_grid(wards, GRID_STEP_KM)
candidates_raw = pd.concat([cand_ward, cand_grid], ignore_index=True)

print(f"Tổng ứng viên sinh ban đầu: {len(candidates_raw)}")
print(f"Tâm phường: {len(cand_ward)}")
print(f"Lưới: {len(cand_grid)}")

2.2 Lọc ứng viên không hợp lệ & vẽ hình minh hoạ
- Loại bỏ các ứng viên quá gần trạm đã có (< D_MIN_KM) và các điểm lưới nằm ngoài vùng xét (cách tâm phường gần nhất > R_SERVICE_KM, ví dụ rơi vào sông, rừng, khu vực thưa dân), sau đó đánh chỉ số id làm phần tử cho state. Hình vẽ lưu ra ../figures/ung_vien.png

In [ ]:
d_to_station = distance_matrix(
    candidates_raw["lat"], candidates_raw["lon"],
    existing["lat"], existing["lon"]
).min(axis=1)

d_to_ward = distance_matrix(
    candidates_raw["lat"], candidates_raw["lon"],
    wards["lat"], wards["lon"]
).min(axis=1)

mask_valid = (d_to_station >= D_MIN_KM) & (d_to_ward <= R_SERVICE_KM)
candidates_removed = candidates_raw[~mask_valid].reset_index(drop=True)
candidates = candidates_raw[mask_valid].reset_index(drop=True)
candidates["id"] = np.arange(len(candidates)) # id = chỉ số dòng, làm phần tử của state

print(f"Đã loại {(~mask_valid).sum()} ứng viên, còn lại {len(candidates)} ứng viên hợp lệ")
print(f"Tâm phường: {(candidates['nguon'] == 'tam_phuong').sum()}")
print(f"Lưới: {(candidates['nguon'] == 'luoi').sum()}")
display(candidates.head())

# Vẽ hình minh hoạ tập ứng viên giữ lại và bị loại
fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(
    candidates_removed["lon"], candidates_removed["lat"],
    s=4, c="lightcoral", alpha=0.25, label=f"Ứng viên bị loại ({len(candidates_removed)})"
)
ax.scatter(
    candidates["lon"], candidates["lat"],
    s=14, c="dimgray", alpha=0.75, label=f"Ứng viên hợp lệ ({len(candidates)})"
)
ax.scatter(
    existing["lon"], existing["lat"],
    marker="s", s=40, c="tab:blue", edgecolor="k", zorder=4,
    label=f"Trạm Metro hiện có / đang xây ({len(existing)})"
)

ax.set_xlabel("Kinh độ")
ax.set_ylabel("Vĩ độ")
ax.set_title("Tập vị trí ứng viên sau khi lọc ràng buộc khoảng cách")
ax.set_aspect(1 / np.cos(np.radians(wards["lat"].mean())))
ax.legend(loc="upper left")
plt.savefig("../figures/ung_vien.png", dpi=150, bbox_inches="tight")
plt.show()

2.3 Tính trước các ma trận khoảng cách
- Tính sẵn các ma trận khoảng cách một lần duy nhất

In [ ]:
POP = wards["dan_so"].to_numpy(dtype=float)
TOTAL_POP = POP.sum()

D_CW = distance_matrix(candidates["lat"], candidates["lon"], wards["lat"], wards["lon"])
D_CC = distance_matrix(candidates["lat"], candidates["lon"], candidates["lat"], candidates["lon"])
D_SW = distance_matrix(existing["lat"], existing["lon"], wards["lat"], wards["lon"])
D_CS = distance_matrix(candidates["lat"], candidates["lon"], existing["lat"], existing["lon"])

COVERED_BY_EXISTING = (D_SW <= R_SERVICE_KM).any(axis=0)
N_CAND = len(candidates)

print("Kích thước D_CW:", D_CW.shape, "| D_CC:", D_CC.shape, "| D_CS:", D_CS.shape)
print(f"Số phường đã được trạm hiện có phục vụ (R <= {R_SERVICE_KM} KM): "
      f"{COVERED_BY_EXISTING.sum()}/{len(wards)} phường")
print(f"Dân số đã được trạm hiện có phục vụ: "
      f"{POP[COVERED_BY_EXISTING].sum():,.0f} / {TOTAL_POP:,.0f} "
      f"({POP[COVERED_BY_EXISTING].sum() / TOTAL_POP * 100:.2f}%)")

## Khối 3: Mô hình hoá bài toán

3.1 Tạo trạng thái ngẫu nhiên hợp (random_state)
- Biểu diễn trạng thái dưới dạng tuple các id ứng viên đã sắp xếp tăng dần. Hàm random_state chọn ngẫu nhiên từng ứng viên sao cho khoảng cách giữa các trạm mới được chọn luôn đạt tối thiểu D_MIN_KM

In [ ]:
def make_state(indices):
    return tuple(sorted(set(int(i) for i in indices)))

def random_state(n_stations, rng):
    chosen = []
    for i in rng.permutation(N_CAND):
        if all(D_CC[i, j] >= D_MIN_KM for j in chosen):
            chosen.append(int(i))
            if len(chosen) == n_stations:
                break
    return make_state(chosen)

3.2 & 3.3 Tính dân số phục vụ, chiều dài tuyến (MST) và chi phí xây dựng
- Dân số phục vụ (served_population): Tổng dân số các phường/xã nằm trong bán kính R_SERVICE_KM của ít nhất một trạm (cũ hoặc mới), mỗi phường chỉ tính một lần
- Chiều dài tuyến (line_length): Dùng thuật toán Prim tìm cây khung nhỏ nhất (MST) nối các trạm mới lại với nhau, cộng thêm đoạn nối ngắn nhất từ cụm trạm mới về mạng lưới trạm hiện có
- Chi phí xây dựng (construction_cost): Tính theo số trạm mới và tổng chiều dài tuyến nối

In [ ]:
def served_population(state):
    covered = COVERED_BY_EXISTING.copy()
    if state:
        covered |= (D_CW[list(state)] <= R_SERVICE_KM).any(axis=0)
    return float(POP[covered].sum())

def line_length(state):
    idx = list(state)
    if not idx:
        return 0.0
    sub = D_CC[np.ix_(idx, idx)]
    n = len(idx)
    in_tree = np.zeros(n, dtype=bool)
    in_tree[0] = True
    best = sub[0].copy()
    total = 0.0
    for _ in range(n - 1):
        best_masked = np.where(in_tree, np.inf, best)
        j = int(np.argmin(best_masked))
        total += best_masked[j]
        in_tree[j] = True
        best = np.minimum(best, sub[j])
    total += D_CS[idx].min()
    return float(total)

def construction_cost(state):
    return float(len(state) * COST_PER_STATION + line_length(state) * COST_PER_KM)

3.4 Kiểm tra ràng buộc và tính hình phạt mềm (penalty)
- Áp dụng hình phạt mềm tăng dần theo mức độ vi phạm 4 ràng buộc: khoảng cách tối thiểu giữa 2 trạm mới (>= D_MIN_KM), khoảng cách từ trạm mới đến trạm hiện có (>= D_MIN_KM), số trạm mới tối đa (<= MAX_NEW_STATIONS), và tổng chi phí (<= BUDGET)

In [ ]:
def penalty(state):
    idx = list(state)
    p = 0.0
    if len(idx) > 1:
        sub = D_CC[np.ix_(idx, idx)]
        iu = np.triu_indices(len(idx), k=1)
        p += np.clip(D_MIN_KM - sub[iu], 0, None).sum() / D_MIN_KM
    if idx:
        p += np.clip(D_MIN_KM - D_CS[idx].min(axis=1), 0, None).sum() / D_MIN_KM
    p += max(0, len(idx) - MAX_NEW_STATIONS)
    p += max(0.0, construction_cost(state) - BUDGET) / BUDGET
    return float(p)

def is_feasible(state):
    return penalty(state) == 0.0

3.5 Hàm mục tiêu tổng hợp (objective & objective_breakdown)
- Chuẩn hoá phần dân số phục vụ theo TOTAL_POP và phần chi phí theo BUDGET để các thành phần cùng thang đo

In [ ]:
def objective(state, weights=DEFAULT_WEIGHTS):
    pop_score = served_population(state) / TOTAL_POP
    cost_score = construction_cost(state) / BUDGET
    return float(
        weights["pop"] * pop_score
        - weights["cost"] * cost_score
        - weights["penalty"] * penalty(state)
    )

def objective_breakdown(state, weights=DEFAULT_WEIGHTS):
    return {
        "so_tram_moi": len(state),
        "dan_so_phuc_vu": served_population(state),
        "ty_le_phuc_vu": served_population(state) / TOTAL_POP,
        "chieu_dai_km": line_length(state),
        "chi_phi": construction_cost(state),
        "hinh_phat": penalty(state),
        "diem": objective(state, weights),
    }

3.6 & 3.7 Sinh toàn bộ hàng xóm (get_neighbors) và hàng xóm ngẫu nhiên (random_neighbor)
- Hỗ trợ 3 toán tử lân cận: thay 1 trạm hiện tại bằng 1 ứng viên bên ngoài (swap), thêm 1 trạm (add), hoặc bớt 1 trạm (remove)

In [ ]:
def get_neighbors(state, allow_add_remove=True):
    current = set(state)
    outside = [c for c in range(N_CAND) if c not in current]
    neighbors = set()
    for s in state:
        for c in outside:
            neighbors.add(make_state((current - {s}) | {c}))
    if allow_add_remove:
        if len(state) < MAX_NEW_STATIONS:
            for c in outside:
                neighbors.add(make_state(current | {c}))
        if len(state) > 1:
            for s in state:
                neighbors.add(make_state(current - {s}))
    neighbors.discard(state)
    return list(neighbors)

def random_neighbor(state, rng, allow_add_remove=True):
    current = set(state)
    outside = [c for c in range(N_CAND) if c not in current]
    moves = []
    if len(state) > 0:
        moves.append("swap")
    if allow_add_remove and len(state) < MAX_NEW_STATIONS:
        moves.append("add")
    if allow_add_remove and len(state) > 1:
        moves.append("remove")
    move = rng.choice(moves)
    if move == "swap":
        s = int(rng.choice(list(state)))
        c = int(rng.choice(outside))
        return make_state((current - {s}) | {c})
    if move == "add":
        return make_state(current | {int(rng.choice(outside))})
    return make_state(current - {int(rng.choice(list(state)))})

Kiểm thử tự động và đo tốc độ

In [ ]:
import time

rng = np.random.default_rng(SEED)

# 1. random_state hợp lệ và lặp lại được theo seed
s1 = random_state(5, np.random.default_rng(1))
s2 = random_state(5, np.random.default_rng(1))
assert s1 == s2 and s1 == tuple(sorted(s1)) and len(s1) <= 5
idx1 = list(s1)
assert all(D_CC[idx1[a], idx1[b]] >= D_MIN_KM for a in range(len(idx1)) for b in range(a + 1, len(idx1)))
assert (D_CS[idx1].min(axis=1) >= D_MIN_KM).all()

# 2. Thêm trạm không làm giảm dân số phục vụ
next_c = next(c for c in range(N_CAND) if c not in s1)
s_add = make_state(set(s1) | {next_c})
assert served_population(s_add) >= served_population(s1)

# 3. Trạng thái rỗng = chỉ tính dân số các trạm hiện có
assert served_population(()) == POP[COVERED_BY_EXISTING].sum()

# 4. Hàng xóm đúng định dạng tuple tăng dần, không trùng chính nó
t0 = time.perf_counter()
nb = get_neighbors(s1)
scores = [objective(x) for x in nb]
dt = time.perf_counter() - t0
assert all(isinstance(x, tuple) and x == tuple(sorted(x)) for x in nb)
assert s1 not in nb and len(nb) == len(set(nb))
print(f"Số hàng xóm của 5 trạm (N_CAND={N_CAND}): {len(nb)} | Thời gian sinh + chấm điểm: {dt:.3f}s")

# 5. random_neighbor luôn nằm trong tập get_neighbors
nb_set = set(nb)
for _ in range(50):
    assert random_neighbor(s1, rng) in nb_set

# 6. Chạy thử leo đồi đơn giản, kiểm tra toàn bộ pipeline
cur = s1
for _ in range(15):
    best = max(get_neighbors(cur), key=objective)
    if objective(best) <= objective(cur):
        break
    cur = best

print("Trạng thái ban đầu:", objective_breakdown(s1))
print("Sau leo đồi thử nghiệm:", objective_breakdown(cur))
assert objective(cur) >= objective(s1)
assert is_feasible(cur)
print("=> TẤT CẢ KIỂM THỬ ĐỀU QUA")